# NB07: Directionality Changes and Loop Detection

Compare reaction directionality under Marvin vs OPAM2 thermodynamic constraints.
Detect thermodynamic loops using pFBA (minimized total flux) and quantify how
loop membership changes with updated deltaG values.

**Directionality**: from FVA with thermo constraints applied.
**Loop detection**: reactions with flux in standard FBA but not pFBA.

In [1]:
import json
from pathlib import Path

import cobra
from cobra.flux_analysis import flux_variability_analysis, pfba
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

## 1. Load data

In [2]:
models = {}
for name in ['ecoli', 'bsubtilis']:
    models[name] = cobra.io.load_json_model(str(DATA_DIR / f'model_{name}.json'))
    print(f'{name}: {len(models[name].reactions)} reactions')

with open(DATA_DIR / 'dg_predictions_marvin.json') as f:
    marvin_dg = json.load(f)
with open(DATA_DIR / 'dg_predictions_opam2.json') as f:
    opam2_dg = json.load(f)

DG_THRESHOLD = 5.0

ecoli: 1765 reactions


bsubtilis: 1345 reactions


## 2. Directionality comparison: Marvin vs OPAM2

In [3]:
def apply_thermo(model, dg_preds, threshold=DG_THRESHOLD):
    constrained = 0
    for rxn in model.reactions:
        rxn_id = rxn.id.replace('_c0', '')
        if rxn_id in dg_preds:
            dg = dg_preds[rxn_id]['dG_mean']
            if dg > threshold:
                rxn.upper_bound = min(rxn.upper_bound, 0)
                constrained += 1
            elif dg < -threshold:
                rxn.lower_bound = max(rxn.lower_bound, 0)
                constrained += 1
    return constrained

def classify_fva(fva_df):
    tol = 1e-9
    classes = {}
    for rxn_id in fva_df.index:
        mn = fva_df.loc[rxn_id, 'minimum']
        mx = fva_df.loc[rxn_id, 'maximum']
        if abs(mn) < tol and abs(mx) < tol:
            classes[rxn_id] = 'blocked'
        elif mn >= -tol and mx > tol:
            classes[rxn_id] = 'forward'
        elif mn < -tol and mx <= tol:
            classes[rxn_id] = 'reverse'
        else:
            classes[rxn_id] = 'variable'
    return classes

direction_records = []

for org_id, model in models.items():
    print(f'\n=== {org_id}: Directionality ===')
    
    model_m = model.copy()
    apply_thermo(model_m, marvin_dg)
    fva_m = flux_variability_analysis(model_m, fraction_of_optimum=0.0)
    dir_m = classify_fva(fva_m)
    
    model_o = model.copy()
    apply_thermo(model_o, opam2_dg)
    fva_o = flux_variability_analysis(model_o, fraction_of_optimum=0.0)
    dir_o = classify_fva(fva_o)
    
    changes = {'forward→reverse': 0, 'reverse→forward': 0, 'variable→forward': 0,
               'variable→reverse': 0, 'forward→variable': 0, 'reverse→variable': 0,
               'other': 0}
    
    for rxn_id in set(dir_m.keys()) & set(dir_o.keys()):
        dm, do = dir_m[rxn_id], dir_o[rxn_id]
        rxn_seed = rxn_id.replace('_c0', '')
        
        direction_records.append({
            'organism': org_id,
            'reaction': rxn_id,
            'marvin_dir': dm,
            'opam2_dir': do,
            'changed': dm != do,
            'marvin_dg': marvin_dg.get(rxn_seed, {}).get('dG_mean'),
            'opam2_dg': opam2_dg.get(rxn_seed, {}).get('dG_mean'),
        })
        
        if dm != do:
            key = f'{dm}→{do}'
            if key in changes:
                changes[key] += 1
            else:
                changes['other'] += 1
    
    total_changed = sum(changes.values())
    print(f'  Total direction changes: {total_changed}')
    for k, v in changes.items():
        if v > 0:
            print(f'    {k}: {v}')

df_dir = pd.DataFrame(direction_records)
print(f'\nTotal records: {len(df_dir)}, changed: {df_dir["changed"].sum()}')


=== ecoli: Directionality ===


  Total direction changes: 5
    other: 5

=== bsubtilis: Directionality ===


  Total direction changes: 0

Total records: 3110, changed: 5


## 3. Loop detection: pFBA comparison

In [4]:
loop_records = []

for org_id, model in models.items():
    print(f'\n=== {org_id}: Loop detection ===')
    
    for label, dg_preds in [('marvin', marvin_dg), ('opam2', opam2_dg)]:
        mdl = model.copy()
        apply_thermo(mdl, dg_preds)
        
        sol_std = mdl.optimize()
        if sol_std.status != 'optimal':
            print(f'  {label}: infeasible')
            continue
        
        sol_pfba = pfba(mdl)
        
        tol = 1e-6
        loop_rxns = []
        for rxn_id in sol_std.fluxes.index:
            if abs(sol_std.fluxes[rxn_id]) > tol and abs(sol_pfba.fluxes[rxn_id]) < tol:
                loop_rxns.append(rxn_id)
        
        print(f'  {label}: {len(loop_rxns)} loop reactions')
        
        for rxn_id in loop_rxns:
            loop_records.append({
                'organism': org_id,
                'thermo_source': label,
                'reaction': rxn_id,
                'std_flux': sol_std.fluxes[rxn_id],
                'pfba_flux': sol_pfba.fluxes[rxn_id],
            })

if loop_records:
    df_loops = pd.DataFrame(loop_records)
    
    for org_id in models:
        m_loops = set(df_loops[(df_loops['organism']==org_id) & (df_loops['thermo_source']=='marvin')]['reaction'])
        o_loops = set(df_loops[(df_loops['organism']==org_id) & (df_loops['thermo_source']=='opam2')]['reaction'])
        print(f'\n{org_id} loop comparison:')
        print(f'  Marvin loops: {len(m_loops)}')
        print(f'  OPAM2 loops: {len(o_loops)}')
        print(f'  New loops (OPAM2 only): {len(o_loops - m_loops)}')
        print(f'  Removed loops (Marvin only): {len(m_loops - o_loops)}')
else:
    df_loops = pd.DataFrame()
    print('No loop reactions detected')


=== ecoli: Loop detection ===


  marvin: 0 loop reactions


  opam2: 0 loop reactions

=== bsubtilis: Loop detection ===


  marvin: 0 loop reactions


  opam2: 0 loop reactions
No loop reactions detected


## 4. Visualize

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Direction change summary
ax = axes[0]
for i, org_id in enumerate(models.keys()):
    sub = df_dir[(df_dir['organism']==org_id) & df_dir['changed']]
    if len(sub) == 0:
        continue
    transitions = sub.apply(lambda r: f"{r['marvin_dir']}→{r['opam2_dir']}", axis=1)
    counts = transitions.value_counts().head(6)
    bars = ax.barh([f'{org_id}: {t}' for t in counts.index], counts.values, color=f'C{i}')
ax.set_xlabel('Count')
ax.set_title('Directionality Changes (Marvin → OPAM2)')

# Loop comparison
ax = axes[1]
if len(df_loops) > 0:
    org_ids = list(models.keys())
    x = np.arange(len(org_ids))
    m_counts = [len(df_loops[(df_loops['organism']==o) & (df_loops['thermo_source']=='marvin')]) for o in org_ids]
    o_counts = [len(df_loops[(df_loops['organism']==o) & (df_loops['thermo_source']=='opam2')]) for o in org_ids]
    ax.bar(x - 0.2, m_counts, 0.35, label='Marvin', color='#2196F3')
    ax.bar(x + 0.2, o_counts, 0.35, label='OPAM2', color='#FF9800')
    ax.set_xticks(x)
    ax.set_xticklabels(org_ids)
    ax.set_ylabel('Loop reactions')
    ax.set_title('Thermodynamic Loop Reactions')
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'directionality_loops.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure to figures/directionality_loops.png')

Saved figure to figures/directionality_loops.png


## 5. Save results

In [6]:
df_dir.to_csv(DATA_DIR / 'directionality_diff.tsv', sep='\t', index=False)
print(f'Saved {len(df_dir)} directionality records')

if len(df_loops) > 0:
    df_loops.to_csv(DATA_DIR / 'loop_diff.tsv', sep='\t', index=False)
    print(f'Saved {len(df_loops)} loop records')

Saved 3110 directionality records
